# Chapter 35 Coding Exam — Packaging, Deployment & Reproducible Research Releases (100 Points)

**Book Reference:** Chapter 35, Section 35.7 (Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Reproducible Research Deliverable | Points |
| :--- | :--- | :---: |
| **Task 35.1** | `batch_inference_with_quarantine(service, payloads)` (Resilient batch scoring + dead-letter quarantine) | 30 |
| **Task 35.2** | `benchmark_inference_latency(service, valid_payload, n_runs=50)` (Mean / p95 / max ms & req/s SLA) | 35 |
| **Task 35.3** | `package_end_to_end_research_release(mch_df, out_dir, seed=42)` (Full model + card + manifest + SHA-256 release) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 35 Exam Grader & Helpers
!pip install -q sharukh-tutor
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import platform
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sharukh_tutor as tutor


def generate_model_card_markdown(meta):
    """Provided helper from To-Do 35.3 for Task 35.3."""
    return (
        f"# Model Card: {meta['name']}\n\n"
        f"## Model Details\n"
        f"- **Name:** {meta['name']}\n"
        f"- **Version:** {meta['version']}\n\n"
        f"## Intended Use\n{meta['intended_use']}\n\n"
        f"## Training Data\n{meta['dataset']}\n\n"
        f"## Evaluation Metrics\n"
        f"- **ROC-AUC:** {float(meta['roc_auc']):.4f}\n"
        f"- **Brier Score:** {float(meta['brier_score']):.4f}\n\n"
        f"## Fairness & Limitations\n{meta['limitations']}\n"
    )


def build_reproducibility_bundle(artifact_path, config_dict, out_dir):
    """Provided helper from To-Do 35.4 for Task 35.3."""
    artifact, out = Path(artifact_path), Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    sha = hashlib.sha256(artifact.read_bytes()).hexdigest()
    manifest = {
        "artifact_name": artifact.name,
        "artifact_bytes": int(artifact.stat().st_size),
        "artifact_sha256": sha,
        "python_version": platform.python_version(),
        "created_utc": datetime.now(timezone.utc).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        ),
    }
    (out / "config.json").write_text(
        json.dumps(config_dict, indent=2), encoding="utf-8"
    )
    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    return manifest


def verify_reproducibility_bundle(out_dir, artifact_path):
    """Provided helper from To-Do 35.5 for Task 35.3."""
    out = Path(out_dir)
    manifest = json.loads(
        (out / "manifest.json").read_text(encoding="utf-8")
    )
    sha = hashlib.sha256(Path(artifact_path).read_bytes()).hexdigest()
    ok_hash = sha == manifest["artifact_sha256"]
    ok_cfg = (out / "config.json").exists()
    return {
        "hash_match": bool(ok_hash),
        "config_present": bool(ok_cfg),
        "bundle_valid": bool(ok_hash and ok_cfg),
    }


tutor.activate(chapter="ch35", notebook="coding_exam")


---
## Exam Task 35.1 (30 Points): `batch_inference_with_quarantine(service, payloads)`

Implement resilient batch scoring with a dead-letter quarantine queue:
1. Iterate `for idx, p in enumerate(payloads):` without mutating `payloads`.
2. Call `service.predict_one(p)` inside `try / except ValueError as exc:`.
3. Append valid prediction dicts to `"succeeded"` and `{"index": idx, "error": str(exc)}` to `"quarantined"`.
4. Compute `success_rate = round(len(succeeded) / len(payloads), 4)` (or `0.0` if `payloads` is empty).
5. Return `{"succeeded": list, "quarantined": list, "success_rate": float}`.

In [ ]:
def batch_inference_with_quarantine(service, payloads):
    """Score valid payloads and quarantine ValueError rows by index."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    succeeded = []
    quarantined = []
    for idx, payload in enumerate(payloads):
        try:
            succeeded.append(service.predict_one(payload))
        except ValueError as exc:
            quarantined.append({"index": idx, "error": str(exc)})
    rate = round(len(succeeded) / len(payloads), 4) if payloads else 0.0
    return {
        "succeeded": succeeded,
        "quarantined": quarantined,
        "success_rate": rate,
    }
    ### END SOLUTION


---
## Exam Task 35.2 (35 Points): `benchmark_inference_latency(service, valid_payload, n_runs=50)`

Benchmark single-request inference latency for clinical SLA reporting:
1. Run `service.predict_one(valid_payload)` exactly `n_runs` times, timing each call with `time.perf_counter()` in milliseconds (`* 1000.0`).
2. Compute `mean_ms`, `p95_ms` (`np.percentile(times_ms, 95)`), `max_ms`, and `throughput_per_sec = 1000.0 / mean_ms`.
3. Return `{"n_runs": int, "mean_ms": round4, "p95_ms": round4, "max_ms": round4, "throughput_per_sec": round2}`.

In [ ]:
def benchmark_inference_latency(service, valid_payload, n_runs=50):
    """Measure mean, p95, max latency (ms) & throughput (req/s)."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    times_ms = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        service.predict_one(valid_payload)
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    mean_ms = float(np.mean(times_ms))
    p95_ms = float(np.percentile(times_ms, 95))
    max_ms = float(np.max(times_ms))
    tps = 1000.0 / max(mean_ms, 1e-9)
    return {
        "n_runs": int(n_runs),
        "mean_ms": round(mean_ms, 4),
        "p95_ms": round(p95_ms, 4),
        "max_ms": round(max_ms, 4),
        "throughput_per_sec": round(tps, 2),
    }
    ### END SOLUTION


---
## Exam Task 35.3 (35 Points): `package_end_to_end_research_release(mch_df, out_dir, seed=42)`

Without mutating `mch_df`, package a complete, peer-review-ready ML release inside `out_dir`:
1. Copy `mch_df`, normalize `residence` and `wealth_quintile` via `.astype(str).str.strip().str.title()`, and replace sentinel `999` with `np.nan` in `num_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]`.
2. Run a stratified 75/25 `train_test_split` (`random_state=seed`) predicting `child_stunted` from `num_cols + ["residence", "wealth_quintile"]`.
3. Fit a standard Scikit-Learn `Pipeline([("prep", ColumnTransformer(...)), ("model", LogisticRegression(max_iter=500))])` (median+scaler numeric, most_frequent+OneHotEncoder(`handle_unknown="ignore", sparse_output=False`) categorical).
4. Compute `test_auc` and `brier`, then write `model.joblib`, `MODEL_CARD.md`, `config.json`, and `manifest.json` inside `out_dir`.
5. Run `verify_reproducibility_bundle(out_dir, model_path)` and return `{"bundle_valid": bool, "test_auc": round4, "files_created": sorted_list}`.

In [ ]:
def package_end_to_end_research_release(mch_df, out_dir, seed=42):
    """Train, package, hash & verify a full MCH research release."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["wealth_quintile"] = (
        work["wealth_quintile"].astype(str).str.strip().str.title()
    )
    num_cols = [
        "maternal_age_years", "anc_visits", "maternal_hb_g_dl"
    ]
    cat_cols = ["residence", "wealth_quintile"]
    work[num_cols] = work[num_cols].replace(999, np.nan)

    X = work[num_cols + cat_cols]
    y = work["child_stunted"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=seed
    )

    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore", sparse_output=False
            )),
        ]), cat_cols),
    ])
    pipe = Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(max_iter=500)),
    ])
    pipe.fit(X_tr, y_tr)

    probs = pipe.predict_proba(X_te)[:, 1]
    test_auc = round(float(roc_auc_score(y_te, probs)), 4)
    brier = round(float(brier_score_loss(y_te, probs)), 4)

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    model_path = out / "model.joblib"
    joblib.dump(pipe, model_path)

    meta = {
        "name": "MCH Child Stunting Risk Classifier",
        "version": "v1.0",
        "intended_use": "Community health screening triage.",
        "dataset": "mch_survey_sample (n=500, 75/25 stratified)",
        "roc_auc": test_auc,
        "brier_score": brier,
        "limitations": "Cross-sectional; prospective audit needed.",
    }
    (out / "MODEL_CARD.md").write_text(
        generate_model_card_markdown(meta), encoding="utf-8"
    )
    cfg = {
        "seed": int(seed),
        "test_size": 0.25,
        "test_auc": test_auc,
        "brier_score": brier,
    }
    build_reproducibility_bundle(model_path, cfg, out)
    ver = verify_reproducibility_bundle(out, model_path)
    files = sorted(p.name for p in out.iterdir() if p.is_file())
    return {
        "bundle_valid": bool(ver["bundle_valid"]),
        "test_auc": test_auc,
        "files_created": files,
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch35_exam_01": batch_inference_with_quarantine,
    "ch35_exam_02": benchmark_inference_latency,
    "ch35_exam_03": package_end_to_end_research_release,
}

tutor.check_exam("ch35", exam_submissions)


---
## Chapter 35 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 35.1 (Empty Batch Guard & Narrow Exception Catching):** Catch only `ValueError` (never a bare `except:` that hides genuine code bugs like `AttributeError`) and guard `if payloads else 0.0` so empty daily batches never crash with `ZeroDivisionError`.
- **Task 35.2 (High-Resolution `perf_counter()` vs. `time.time()`):** Always benchmark latency with monotonic `time.perf_counter()` rather than wall-clock `time.time()`, and report `p95_ms` alongside `mean_ms` to capture garbage-collection or cold-cache tail spikes.
- **Task 35.3 (Portable Standard Pipelines & Self-Verifying Releases):** Using built-in Scikit-Learn steps (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `LogisticRegression`) guarantees `model.joblib` reloads cleanly in any container without custom `__main__` class pickling failures, while calling `verify_reproducibility_bundle` immediately after writing catches truncation or hash mismatches before release.